# Moduł 1/3 — przygotowanie danych EEGMMIDB (wspólne dla wszystkich algorytmów)

**Cel:** jednokrotnie wczytać EDF, przefiltrować sygnał, wyznaczyć epoki, a następnie zapisać **niezależny od modelu** zbiór danych i **stałe podziały uczestników** do walidacji LOSO.

Pliki wyjściowe (w `dane_eegmmidb/zestawy/<identyfikator>/`):

| Plik | Znaczenie |
|---|---|
| `X.npy` | `float32`, `[n_epok, n_kanałów, n_próbek]`, sygnał EEG w **woltach** |
| `y.npy` | `int8`, etykiety `1=T1`, `2=T2` |
| `groups.npy` | `int16`, numer uczestnika dla każdej epoki |
| `runs.npy` | `int8`, numer runu dla każdej epoki |
| `manifest.json` | schemat, metadane, parametry preprocessingowe, checksums SHA-256 |
| `podzialy_loso.json` | ten sam trening/walidacja/test **po osobach** dla wszystkich algorytmów |
| `uczestnicy.csv` | liczba epok każdej osoby i każdej klasy |

Każdy plik `.npy` ma wspólną, ustaloną kolejność epok: **rosnący numer uczestnika → kolejność RUNS → kolejność zdarzeń w runie**. Cache jednego uczestnika jest współdzielony między trybami TEST_20 i FULL_109. **Nie ma tu CSP, LDA ani uczenia modelu.**

**Uruchamianie:** notebooki 1, 2 i 3 uruchamiaj z tego samego katalogu roboczego (np. wszystkie trzy pliki w jednym folderze).

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import warnings

import mne
import numpy as np
import pandas as pd
from mne.datasets import eegbci
from sklearn.model_selection import GroupShuffleSplit
import sklearn

# ===== KONFIGURACJA DANYCH (zmieniaj tutaj) =====
MODE = "FULL_109"  # "TEST_20" albo "FULL_109"
SUBJECT_IDS = list(range(1, 21)) if MODE == "TEST_20" else list(range(1, 110))

# To zadanie MIESZANE: 3/7/11 = motor execution, 4/8/12 = motor imagery.
RUNS = [3, 4, 7, 8, 11, 12]
DATA_ROOT = "/home/eliza/physionet.org/files/eegmmidb/1.0.0"
# DATA_ROOT = None  # MNE może pobierać EDF z internetu

FMIN, FMAX = 8.0, 30.0
TMIN, TMAX = 1.0, 3.5
TARGET_SFREQ = 128.0
EEG_CHANNELS = None  

VALIDATION_SIZE = 0.20
SEED = 42

DATA_BASE = Path.cwd() / "dane_eegmmidb"
USE_CACHE = True
FORCE_REBUILD_SUBJECT_CACHE = False
SHOW_PREVIEW_PLOTS = False
SHOW_RAW_BROWSER = False
PREPROCESSING_VERSION = 1 

assert MODE in {"TEST_20", "FULL_109"}
assert len(SUBJECT_IDS) == len(set(SUBJECT_IDS)) and len(SUBJECT_IDS) >= 3
assert all(1 <= s <= 109 for s in SUBJECT_IDS)
assert 0 < VALIDATION_SIZE < 1
assert 0 < FMIN < FMAX < TARGET_SFREQ / 2
assert TMIN < TMAX
SUBJECT_IDS = sorted(SUBJECT_IDS)
print("Python:", platform.python_version(), "| MNE:", mne.__version__,
      "| scikit-learn:", sklearn.__version__)
print(f"Tryb {MODE}: {len(SUBJECT_IDS)} uczestników")

Python: 3.12.12 | MNE: 1.13.2 | scikit-learn: 1.9.1
Tryb FULL_109: 109 uczestników


In [2]:
HAND_LEFT_RIGHT = {3, 4, 7, 8, 11, 12}
HAND_FEET = {5, 6, 9, 10, 13, 14}
MOTOR_IMAGERY = {4, 6, 8, 10, 12, 14}
MOTOR_EXECUTION = {3, 5, 7, 9, 11, 13}


def validate_runs(runs):
    runs = [int(x) for x in runs]
    if not runs or len(runs) != len(set(runs)):
        raise ValueError("RUNS musi być niepustą listą bez duplikatów.")
    if set(runs) <= HAND_LEFT_RIGHT:
        family, names = "left_right", {1: "lewa dłoń", 2: "prawa dłoń"}
    elif set(runs) <= HAND_FEET:
        family, names = "hands_feet", {1: "obie dłonie", 2: "obie stopy"}
    else:
        raise ValueError("Nie mieszaj runów lewa/prawa z runami obie dłonie/stopy.")
    executed = sorted(set(runs) & MOTOR_EXECUTION)
    if executed:
        warnings.warn(f"Wybrane runy {executed} to MOTOR EXECUTION, a nie motor imagery.")
    return family, names


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def stable_hash(payload, size=12):
    return hashlib.sha256(json.dumps(
        payload, sort_keys=True, ensure_ascii=False,
    ).encode("utf-8")).hexdigest()[:size]


TASK_FAMILY, CLASS_NAMES = validate_runs(RUNS)
PREP_CONFIG = {
    "runs": list(RUNS), "family": TASK_FAMILY,
    "fmin": FMIN, "fmax": FMAX, "tmin": TMIN, "tmax": TMAX,
    "target_sfreq": TARGET_SFREQ, "channels": EEG_CHANNELS,
    "reference": "average", "filter_method": "fir", "filter_design": "firwin",
    "baseline": None, "reject_by_annotation": True,
    "preprocessing_version": PREPROCESSING_VERSION,
}
PREP_HASH = stable_hash(PREP_CONFIG)
CACHE_DIR = DATA_BASE / "cache" / PREP_HASH
print("Protokół:", TASK_FAMILY, "| T1:", CLASS_NAMES[1], "| T2:", CLASS_NAMES[2])
print("Id preprocessing:", PREP_HASH)

Protokół: left_right | T1: lewa dłoń | T2: prawa dłoń
Id preprocessing: ac6a0de9ee1b


/tmp/ipykernel_10949/2811430649.py:19: UserWarning: Wybrane runy [3, 7, 11] to MOTOR EXECUTION, a nie motor imagery.
  warnings.warn(f"Wybrane runy {executed} to MOTOR EXECUTION, a nie motor imagery.")


## 2. Odczyt EDF → referencja → filtracja → resampling → epoki

Metoda zgodna z notebookiem źródłowym: standaryzacja nazw kanałów, montaż `standard_1005`, średnia referencja EEG, filtr 8–30 Hz FIR (lub inne wartości z konfiguracji), resampling do 128 Hz, epoki `[1.0, 3.5]` sekundy względem `T1/T2`, bez korekty bazowej. Filtracja odbywa się przed podziałem **bez uczenia parametrów z etykiet**. W preprocessing nie ma żadnego wybierania kanałów ani parametrów na podstawie wyników testowych.

In [ ]:
def get_edf_path(subject, run):
    if DATA_ROOT is None:
        return Path(eegbci.load_data(subject, [run], update_path=False, verbose=False)[0])
    name = f"S{subject:03d}"
    path = Path(DATA_ROOT).expanduser() / name / f"{name}R{run:02d}.edf"
    if not path.is_file():
        raise FileNotFoundError(
            f"Brak EDF: {path}. Zmień DATA_ROOT lub ustaw DATA_ROOT=None."
        )
    return path


def load_raw(subject, run):
    raw = mne.io.read_raw_edf(str(get_edf_path(subject, run)), preload=True, verbose=False)
    eegbci.standardize(raw)
    raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
                    on_missing="warn", verbose=False)
    return raw


def preprocess_raw(raw):
    out = raw.copy()
    if EEG_CHANNELS is None:
        out.pick("eeg")
    else:
        missing = sorted(set(EEG_CHANNELS) - set(out.ch_names))
        if missing:
            raise ValueError(f"Nie znaleziono kanałów: {missing}")
        out.pick(EEG_CHANNELS)
    out.set_eeg_reference(ref_channels="average", verbose=False)
    out.filter(FMIN, FMAX, method="fir", fir_design="firwin", verbose=False)
    if not np.isclose(out.info["sfreq"], TARGET_SFREQ):
        out.resample(TARGET_SFREQ, npad="auto", verbose=False)
    return out


def raw_to_epochs(filtered_raw):
    events, _ = mne.events_from_annotations(
        filtered_raw, event_id={"T1": 1, "T2": 2}, verbose=False
    )
    epochs = mne.Epochs(
        filtered_raw, events, event_id={"T1": 1, "T2": 2},
        tmin=TMIN, tmax=TMAX, baseline=None, picks="eeg",
        preload=True, reject_by_annotation=True, verbose=False,
    )
    if len(epochs) == 0:
        raise ValueError("Po epokowaniu nie pozostała żadna epoka T1/T2.")
    return epochs


def check_both_classes(y, location):
    if set(np.unique(y)) != {1, 2}:
        raise ValueError(f"{location}: wymagane T1/T2, znaleziono {np.unique(y)}")


def read_subject_npz(path, subject):
    with np.load(path, allow_pickle=False) as data:
        result = {"X": data["X"], "y": data["y"],
                  "runs": data["runs"], "ch_names": data["ch_names"].tolist()}
    X, y, runs = result["X"], result["y"], result["runs"]
    if X.ndim != 3 or X.dtype != np.float32 or y.dtype != np.int8 or runs.dtype != np.int8:
        raise ValueError(f"S{subject:03d}: nieprawidłowa struktura cache: {path}")
    if not (X.shape[0] == len(y) == len(runs)) or X.shape[1] != len(result["ch_names"]):
        raise ValueError(f"S{subject:03d}: niezgodne wymiary cache: {path}")
    if not np.isfinite(X).all() or not set(np.unique(runs)).issubset(set(RUNS)):
        raise ValueError(f"S{subject:03d}: błędne wartości w cache: {path}")
    check_both_classes(y, f"S{subject:03d}")
    return result


def prepare_subject(subject):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path = CACHE_DIR / f"S{subject:03d}.npz"
    if USE_CACHE and path.exists() and not FORCE_REBUILD_SUBJECT_CACHE:
        return read_subject_npz(path, subject), path

    X_parts, y_parts, run_parts = [], [], []
    channel_names = None
    for run in RUNS:
        filtered = preprocess_raw(load_raw(subject, run))
        epochs = raw_to_epochs(filtered)
        X = epochs.get_data(copy=True).astype(np.float32, copy=False)
        y = epochs.events[:, -1].astype(np.int8, copy=True)
        if channel_names is None:
            channel_names = list(epochs.ch_names)
        elif list(epochs.ch_names) != channel_names:
            raise ValueError(f"S{subject:03d}: kolejność kanałów różni się między runami")
        X_parts.append(X)
        y_parts.append(y)
        run_parts.append(np.full(len(y), run, dtype=np.int8))
        del filtered, epochs

    result = {"X": np.concatenate(X_parts), "y": np.concatenate(y_parts),
              "runs": np.concatenate(run_parts), "ch_names": channel_names}
    check_both_classes(result["y"], f"S{subject:03d}")
    if USE_CACHE:
        np.savez_compressed(path, X=result["X"], y=result["y"],
                            runs=result["runs"],
                            ch_names=np.asarray(channel_names, dtype="U32"))
    else:
        np.savez_compressed(path, X=result["X"], y=result["y"],
                            runs=result["runs"],
                            ch_names=np.asarray(channel_names, dtype="U32"))
    return result, path

## 3. Opcjonalny podgląd jednego runu

In [4]:
if SHOW_PREVIEW_PLOTS:
    import matplotlib.pyplot as plt

    subject, run = SUBJECT_IDS[0], RUNS[0]
    raw = load_raw(subject, run)
    print(f"Podgląd S{subject:03d} / R{run:02d}", raw.info)
    if SHOW_RAW_BROWSER:
        raw.plot(n_channels=20, duration=8, block=False)
    raw.compute_psd(fmin=1, fmax=40, picks="eeg", verbose=False).plot(show=False)
    plt.show()
    epochs = raw_to_epochs(preprocess_raw(raw))
    print("Podgląd epok:", epochs.get_data(copy=False).shape)
    if "C3" in epochs.ch_names:
        mne.viz.plot_compare_evokeds(
            {CLASS_NAMES[1]: epochs["T1"].average(),
             CLASS_NAMES[2]: epochs["T2"].average()},
            picks="C3", title="Średnia odpowiedź C3 — podgląd", show=False,
        )
        plt.show()
    del raw, epochs

## 4. Cache per uczestnik + kontrola jednolitej struktury

In [5]:
subject_info = []
expected_shape, expected_channels = None, None
for pos, subject in enumerate(SUBJECT_IDS, 1):
    one_subject, cache_path = prepare_subject(subject)
    X, y = one_subject["X"], one_subject["y"]
    if expected_shape is None:
        expected_shape, expected_channels = list(X.shape[1:]), list(one_subject["ch_names"])
    if list(X.shape[1:]) != expected_shape or list(one_subject["ch_names"]) != expected_channels:
        raise ValueError(f"S{subject:03d}: liczba/przydział kanałów lub próbek różni się od wzorca")
    counts = np.bincount(y, minlength=3)
    subject_info.append({
        "subject_id": int(subject), "n_epochs": int(len(y)),
        "n_T1": int(counts[1]), "n_T2": int(counts[2]),
        "cache_sha256": sha256_file(cache_path),
    })
    print(f"[{pos:3d}/{len(SUBJECT_IDS)}] S{subject:03d}: {len(y)} epok "
          f"(T1={counts[1]}, T2={counts[2]})")
    del one_subject

if expected_shape[0] < 2:
    raise ValueError("Za mało kanałów EEG.")

DATASET_SPEC = {
    "preprocessing_hash": PREP_HASH,
    "subject_ids": SUBJECT_IDS,
    "validation_size": VALIDATION_SIZE,
    "seed": SEED,
    "subject_files": [x["cache_sha256"] for x in subject_info],
    "schema_version": "1.0",
}
DATASET_ID = stable_hash(DATASET_SPEC)
DATASET_DIR = DATA_BASE / "zestawy" / DATASET_ID
MANIFEST_PATH = DATASET_DIR / "manifest.json"
print("Zestaw danych:", DATASET_ID)
print("Kształt pojedynczej epoki [kanały, próbki]:", expected_shape)
print("Docelowy folder:", DATASET_DIR)

/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' i

[  1/109] S001: 90 epok (T1=46, T2=44)


/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' i

[  2/109] S002: 90 epok (T1=46, T2=44)


/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' i

[  3/109] S003: 90 epok (T1=45, T2=45)


... (208 outputs left out)

/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' i

[108/109] S108: 90 epok (T1=43, T2=47)


/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage(mne.channels.make_standard_montage("standard_1005"),
/tmp/ipykernel_10949/2893859135.py:16: FutureWarning: Montage name 'standard_1005' i

[109/109] S109: 90 epok (T1=46, T2=44)
Zestaw danych: 0a8da1dbe454
Kształt pojedynczej epoki [kanały, próbki]: [64, 321]
Docelowy folder: /home/wsl/phd/01_przygotowanie_danych_EEGMMIDB/dane_eegmmidb/zestawy/0a8da1dbe454


## 5. Wspólne podziały LOSO (zapisywane tylko raz)

In [ ]:
def make_splits(subject_ids):
    subject_ids = sorted(map(int, subject_ids))
    folds = []
    for number, test_subject in enumerate(subject_ids, 1):
        pool = np.asarray([s for s in subject_ids if s != test_subject], dtype=np.int16)
        gss = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_SIZE,
                                random_state=SEED + test_subject)
        tr_idx, va_idx = next(gss.split(np.zeros(len(pool)), groups=pool))
        train = sorted(map(int, pool[tr_idx]))
        val = sorted(map(int, pool[va_idx]))
        test = [int(test_subject)]
        assert set(train).isdisjoint(val) and set(train).isdisjoint(test)
        assert set(val).isdisjoint(test)
        assert set(train + val + test) == set(subject_ids)
        folds.append({"fold": number, "test_subject": test_subject,
                      "train_subjects_internal": train,
                      "validation_subjects": val,
                      "test_subjects": test})
    return folds

folds = make_splits(SUBJECT_IDS)
assert folds == make_splits(SUBJECT_IDS) 
assert [f["test_subject"] for f in folds] == SUBJECT_IDS
print("Liczba podziałów:", len(folds), "| przykład:", folds[0])

Liczba podziałów: 109 | przykład: {'fold': 1, 'test_subject': 1, 'train_subjects_internal': [5, 6, 8, 9, 12, 14, 16, 18, 19, 20, 21, 23, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 58, 59, 60, 61, 62, 63, 66, 67, 68, 69, 70, 71, 73, 74, 75, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 88, 89, 90, 91, 92, 93, 94, 95, 97, 98, 99, 100, 102, 104, 105, 106, 107, 108, 109], 'validation_subjects': [2, 3, 4, 7, 10, 11, 13, 15, 17, 22, 24, 35, 46, 57, 64, 65, 72, 82, 87, 96, 101, 103], 'test_subjects': [1]}


## 6. Zapis danych w formacie `.npy` + manifest

In [7]:
def verify_existing_dataset(manifest_path):
    with manifest_path.open(encoding="utf-8") as f:
        manifest = json.load(f)
    if manifest.get("dataset_id") != DATASET_ID or manifest.get("dataset_spec") != DATASET_SPEC:
        raise ValueError("Istniejący manifest nie pasuje do bieżących ustawień.")
    for file_info in manifest["files"].values():
        path = manifest_path.parent / file_info["path"]
        if not path.is_file() or sha256_file(path) != file_info["sha256"]:
            raise ValueError(f"Uszkodzony/zmieniony zbiór danych: {path}")
    return manifest


if MANIFEST_PATH.exists():
    manifest = verify_existing_dataset(MANIFEST_PATH)
    print("Wykorzystano istniejący, zweryfikowany zestaw:", DATASET_DIR)
else:
    if DATASET_DIR.exists():
        raise RuntimeError(
            f"W folderze {DATASET_DIR} są pozostałości nieukończonego zapisu. "
            "Usuń TYLKO ten niekompletny folder i uruchom tę komórkę ponownie."
        )
    DATASET_DIR.mkdir(parents=True, exist_ok=False)
    total_epochs = sum(item["n_epochs"] for item in subject_info)
    n_channels, n_times = expected_shape
    shapes = {
        "X": ((total_epochs, n_channels, n_times), np.float32),
        "y": ((total_epochs,), np.int8),
        "groups": ((total_epochs,), np.int16),
        "runs": ((total_epochs,), np.int8),
    }
    arrays = {
        key: np.lib.format.open_memmap(DATASET_DIR / f"{key}.npy", mode="w+",
                                       dtype=dtype, shape=shape)
        for key, (shape, dtype) in shapes.items()
    }
    offset = 0
    for item in subject_info:
        subject = item["subject_id"]
        data = read_subject_npz(CACHE_DIR / f"S{subject:03d}.npz", subject)
        length = len(data["y"])
        sl = slice(offset, offset + length)
        arrays["X"][sl] = data["X"]
        arrays["y"][sl] = data["y"]
        arrays["groups"][sl] = subject
        arrays["runs"][sl] = data["runs"]
        offset += length
        del data
    assert offset == total_epochs
    for array in arrays.values():
        array.flush()
    del arrays

    split_path = DATASET_DIR / "podzialy_loso.json"
    with split_path.open("w", encoding="utf-8") as f:
        json.dump({"dataset_id": DATASET_ID, "seed": SEED,
                   "validation_size": VALIDATION_SIZE, "folds": folds},
                  f, indent=2, ensure_ascii=False)
    summary_path = DATASET_DIR / "uczestnicy.csv"
    pd.DataFrame(subject_info).to_csv(summary_path, index=False)

    paths = {key: f"{key}.npy" for key in ("X", "y", "groups", "runs")}
    paths["splits"] = "podzialy_loso.json"
    paths["subjects"] = "uczestnicy.csv"
    file_entries = {key: {"path": filename,
                          "sha256": sha256_file(DATASET_DIR / filename)}
                    for key, filename in paths.items()}
    manifest = {
        "schema_version": "1.0", "dataset_id": DATASET_ID, "mode": MODE,
        "dataset_spec": DATASET_SPEC, "subject_ids": SUBJECT_IDS,
        "preprocessing_hash": PREP_HASH, "preprocessing": PREP_CONFIG,
        "n_epochs": total_epochs,
        "epoch_shape": [n_channels, n_times],
        "X_shape": [total_epochs, n_channels, n_times],
        "X_dtype": "float32", "signal_unit": "V",
        "order": "subject ascending -> RUNS order -> event order",
        "channels": expected_channels,
        "labels": {"1": CLASS_NAMES[1], "2": CLASS_NAMES[2]},
        "positive_label": 2,
        "task_types": sorted(
            ({"motor_imagery"} if set(RUNS) & MOTOR_IMAGERY else set()) |
            ({"motor_execution"} if set(RUNS) & MOTOR_EXECUTION else set())
        ),
        "subjects": subject_info,
        "files": file_entries,
        "versions": {"python": platform.python_version(), "mne": mne.__version__,
                     "numpy": np.__version__, "sklearn": sklearn.__version__},
    }
    with MANIFEST_PATH.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    print("Zapisano kompletny zestaw:", DATASET_DIR)

# Szybka kontrola typów, struktury i zgodności liczby epok.
X_check = np.load(DATASET_DIR / manifest["files"]["X"]["path"], mmap_mode="r")
y_check = np.load(DATASET_DIR / manifest["files"]["y"]["path"], mmap_mode="r")
groups_check = np.load(DATASET_DIR / manifest["files"]["groups"]["path"], mmap_mode="r")
runs_check = np.load(DATASET_DIR / manifest["files"]["runs"]["path"], mmap_mode="r")
assert list(X_check.shape) == manifest["X_shape"] and X_check.dtype == np.float32
assert len(X_check) == len(y_check) == len(groups_check) == len(runs_check)
assert y_check.dtype == np.int8 and groups_check.dtype == np.int16 and runs_check.dtype == np.int8
assert np.array_equal(np.unique(groups_check), SUBJECT_IDS)
assert set(np.unique(y_check)) == {1, 2}
assert set(np.unique(runs_check)) == set(RUNS)
assert all(np.all(groups_check[np.flatnonzero(groups_check == s)] == s) for s in SUBJECT_IDS)
del X_check, y_check, groups_check, runs_check

DATA_BASE.mkdir(parents=True, exist_ok=True)
POINTER_PATH = DATA_BASE / "ostatni_zestaw.json"
with POINTER_PATH.open("w", encoding="utf-8") as f:
    json.dump({"manifest_relative_path": str(MANIFEST_PATH.relative_to(DATA_BASE)),
               "dataset_id": DATASET_ID}, f, indent=2)

print("\nGOTOWE — pliki dla pozostałych algorytmów:")
print("Manifest:", MANIFEST_PATH)
print("LOSO:", DATASET_DIR / "podzialy_loso.json")
print("X [epoki, kanały, próbki] =", manifest["X_shape"])
print("Wskazanie bieżącego zestawu:", POINTER_PATH)

Zapisano kompletny zestaw: /home/wsl/phd/01_przygotowanie_danych_EEGMMIDB/dane_eegmmidb/zestawy/0a8da1dbe454

GOTOWE — pliki dla pozostałych algorytmów:
Manifest: /home/wsl/phd/01_przygotowanie_danych_EEGMMIDB/dane_eegmmidb/zestawy/0a8da1dbe454/manifest.json
LOSO: /home/wsl/phd/01_przygotowanie_danych_EEGMMIDB/dane_eegmmidb/zestawy/0a8da1dbe454/podzialy_loso.json
X [epoki, kanały, próbki] = [9844, 64, 321]
Wskazanie bieżącego zestawu: /home/wsl/phd/01_przygotowanie_danych_EEGMMIDB/dane_eegmmidb/ostatni_zestaw.json


## Co przekazywać innym algorytmom?

Zawsze wczytuj `X.npy`, `y.npy`, `groups.npy`, `runs.npy`, `manifest.json` i `podzialy_loso.json`. Nie modyfikuj oryginalnych tablic. Dla modeli wymagających np. innej reprezentacji (`2D`, obrazy, tensory z dodatkową osią) **przekształcaj je dopiero w ich własnym notebooku** i zapisuj identyfikator `dataset_id` w wynikach. Zewnętrzne foldy LOSO i wewnętrzne przydziały osób muszą pozostać identyczne; wszelkie uczenie parametrów na danych (np. skalowanie, PCA, dobór cech) wykonuj tylko na danych treningowych danego folda.